## (1) Load model

In [1]:
from mamba2 import Mamba2, ModelArgs_Mamba2
from transformers import AutoTokenizer

# One of:

#     'state-spaces/mamba2-2.7b'
#     'state-spaces/mamba2-1.3b'
#     'state-spaces/mamba2-780m'
#     'state-spaces/mamba2-370m'
#     'state-spaces/mamba2-130m'
# pretrained_model_name = 'state-spaces/mamba2-130m'
pretrained_model_name = 'state-spaces/mamba2-1.3b'

tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')
# tokenizer.pad_token_id = tokenizer.eos_token_id # TODO, adopted from mamba2-minimal, need to check the vanilla implementation
model, params = Mamba2.from_pretrained(pretrained_model_name, tokenizer=tokenizer, print_config=False) # TODO, set to False later

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


## (2) Generate Text

In [2]:
import jax
import jax.numpy as np

def jax_generate(model,
                 params, 
                 tokenizer,
                 prompt: str,
                 n_tokens_to_gen: int = 50,
                 sample: bool = True,
                 top_k: int = 40,
                 rng = jax.random.PRNGKey(7),
                 ):
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids # In pytorch format
    input_ids = np.array(input_ids.numpy()) # In jax format

    for token_n in range(n_tokens_to_gen):
        indices_to_input = input_ids
        # next_token_logits = model.apply(params, indices_to_input)[:, -1]
        next_token_logits = model.apply(params, indices_to_input)[0][:, -1]

        probs = jax.nn.softmax(next_token_logits, axis=-1)

        if top_k is not None:
            (values, indices) = jax.lax.top_k(probs, k=top_k)
            mask = probs < np.expand_dims(values[:, -1], axis=1)
            probs = np.where(mask, 0.0, probs)
            probs = probs / probs.sum(axis=1, keepdims=True)

        if sample:
            # TODO, might not be 100% correct. 
            rng, subrng = jax.random.split(rng)
            next_indices = jax.random.categorical(subrng, jax.nn.log_softmax(probs), 1, shape=probs.shape[:-1]+(1,))
        else:
            next_indices = np.argmax(probs, axis=-1, keepdims=True)

        input_ids = np.concatenate([input_ids, next_indices], axis=1)
    
    output_completions = [tokenizer.decode(output.tolist()) for output in input_ids][0]

    return output_completions

In [3]:
# import jax
# import jax.numpy as np

# from typing import cast
# from mamba2 import InferenceCache

# def jax_generate(model,
#                  params, 
#                  tokenizer,
#                  prompt: str,
#                  n_tokens_to_gen: int = 50,
#                  temperature: float = 1.0,
#                  sample: bool = True,
#                  top_k: int = 40,
#                  rng = jax.random.PRNGKey(7),
#                  top_p: float = 1.0, # TODO, new
#                  eos_token_id: int = 0, # TODO, new
#                  ):
#     input_ids = tokenizer(prompt, return_tensors='pt').input_ids # In pytorch format
#     input_ids = np.array(input_ids.numpy()) # In jax format
#     prefix, tokens = input_ids[:-1], input_ids[-1:].unsqueeze(0)

#     n_chunked = (prefix.shape[0] // model.args.chunk_size) * model.args.chunk_size

#     if n_chunked > 0:
#         _, h = model(prefix[:n_chunked].unsqueeze(0), None)
#     else:
#         h = [
#             InferenceCache.alloc(1, model.args)
#             for _ in range(model.args.n_layer)
#         ]
#     for i in range(n_chunked, prefix.shape[0]):
#         _, h = model(prefix[i : i + 1].unsqueeze(0), h)

#     # Generate
#     for _ in range(n_tokens_to_gen):
#         out, h = model.apply(params, tokens, h)
#         logits = out[0, -1]
#         if temperature != 1.0:
#             logits = logits / temperature
#         if top_k > 0:
#             indices_to_remove = logits < jax.lax.topk(logits, k=top_k)[0][-1]
#             logits[indices_to_remove] = -np.inf
#         if top_p < 1.0:
#             sorted_logits, sorted_indices = np.sort(logits, descending=True)
#             cum_probs = np.cumsum(jax.nn.softmax(sorted_logits, dim=-1), dim=-1)
#             sorted_indices_to_remove = cum_probs > 0.5
#             sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
#             sorted_indices_to_remove[0] = False
#             indices_to_remove = sorted_indices[sorted_indices_to_remove]
#             logits[indices_to_remove] = -np.inf
#         probs = jax.nn.softmax(logits, dim=-1)
#         # TODO
#         next_token = torch.multinomial(probs, num_samples=1)
#         if next_token.item() == eos_token_id:
#             return
#         tokens = next_token.unsqueeze(0)
#         yield cast(int, next_token.item()), h


In [4]:
sample=False

In [5]:
print(jax_generate(model, params, tokenizer, 'Mamba is the', sample=sample))

Mamba is the first of the new generation of the Mamba family of high-performance, high-capacity, high-reliability, high-density, high-speed, high-performance, high-density, high-speed, high-capacity, high-


In [6]:
print(jax_generate(model, params, tokenizer, 'John: Hi!\nSally:', sample=sample))

John: Hi!
Sally: Hi!
John: So, I'm John.
Sally: I'm Sally.
John: And I'm John.
Sally: And I'm Sally.
John: And I'm John.
Sally: And I


In [7]:
print(jax_generate(model, params, tokenizer, 'The meaning of life is ', sample=sample))

The meaning of life is 
to be happy.
And happiness is not a feeling.
It's a state of mind.
And it's a state of mind that
can be achieved by anyone.
And it's a state of mind that
can be achieved


In [8]:
print(jax_generate(model, params, tokenizer, 'def reverse_string(', sample=sample))

def reverse_string(s):
    return ''.join(s[::-1])

def reverse_list(l):
    return [l[::-1] for i in range(len(l))]

def reverse_dict
